### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [7]:
import pandas as pd
import re
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: artista principal + invitados que vienen en el título
def armar_canasta(fila):
    artistas = [fila['Artist'].strip()]   # el artista principal
    invitados = re.findall(r'[\(\[](?:feat\.?|ft\.?|with)\s+([^\)\]]+)[\)\]]', str(fila['Track']), flags=re.I)
    for grupo in invitados:
        artistas += [a.strip() for a in re.split(r',|&| and ', grupo) if a.strip()]
    return list(dict.fromkeys(artistas))   # quita duplicados

lista_colaboraciones = df_spotify.apply(armar_canasta, axis=1).tolist()

colabs = [c for c in lista_colaboraciones if len(c) > 1]
print(f"Canciones con colaboración: {len(colabs)}")
for c in colabs[:5]:
    print(c)

Canciones con colaboración: 540
['Central Cee', 'Lil Baby']
['Post Malone', 'Morgan Wallen']
['Taylor Swift', 'Post Malone']
['The Kid LAROI', 'Justin Bieber']
['Rema', 'Selena Gomez']


In [8]:

# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit(lista_colaboraciones).transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)

print(df_transacciones.shape)   # (4595, n_artistas)

(4595, 2203)


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [9]:
# --- RETO PARA EL ALUMNO AQUÍ ---

itemsets_frecuentes = apriori(df_transacciones, min_support=0.0005, use_colnames=True)

reglas_musicales = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.2)

print(f"Reglas encontradas: {len(reglas_musicales)}")
display(reglas_musicales.sort_values('confidence', ascending=False)
        [['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Reglas encontradas: 32


,antecedents,consequents,support,confidence,lift
5,frozenset({Bryant Myers}),frozenset({Dei V}),0.000653,1.000000,919.000000
15,frozenset({PARTYNEXTDOOR}),frozenset({Drake}),0.000653,0.750000,40.544118
4,frozenset({Dei V}),frozenset({Bryant Myers}),0.000653,0.600000,919.000000
18,frozenset({Sexyy Red}),frozenset({Drake}),0.000653,0.500000,27.029412
7,frozenset({DJ Khaled}),frozenset({Drake}),0.000653,0.428571,23.168067
9,frozenset({DJ Snake}),frozenset({Ozuna}),0.000653,0.333333,85.092593
3,frozenset({Metro Boomin}),frozenset({21 Savage}),0.000871,0.250000,37.056452
25,frozenset({Lil Durk}),frozenset({Lil Baby}),0.000653,0.214286,28.132653
29,frozenset({Metro Boomin}),frozenset({Travis Scott}),0.000653,0.187500,20.036337
30,frozenset({Young Thug}),frozenset({Travis Scott}),0.000653,0.176471,18.857729


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** Metro Boomin y 21 Savage, con soporte de 0.00087: aparecen juntos en 4 de las 4,595 canciones más escuchadas (~0.09%). Para el negocio, es la colaboración que más se repite entre los éxitos. Es volumen probado, aunque el número absoluto es bajo porque la mayoría de los éxitos son de un solo artista

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** De los éxitos donde aparece PARTYNEXTDOOR, el 75% también tienen a Drake. Si contratamos a PARTYNEXTDOOR, hay un 75% de probabilidad de que Drake deba estar en el track para replicar el éxito que registramos.

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** El lift más alto es Bryant Myers, Dei V (919). El lift compara qué tan seguido aparecen juntos contra lo que se esperaría si fuera pura casualidad.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** PARTYNEXTDOOR + Drake.confianza de 75% , lift de 40.5 y un soporte de 3 éxitos.